# 📦 Notebook 1 — Setup & Exploration des Données
**ISIC Project | Segmentation de Lésions Cutanées**

Ce notebook couvre :
- ✅ Connexion Google Drive
- ✅ Installation des dépendances
- ✅ Exploration et visualisation des données
- ✅ Statistiques sur les images et masques

## 🔗 1.1 — Connexion Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive connecté !')

## 📁 1.2 — Chemins du projet

In [ ]:
import os, sys
from pathlib import Path

PROJECT_PATH  = '/content/drive/MyDrive/ISIC_Project'
DATA_PATH     = os.path.join(PROJECT_PATH, 'data')
IMAGES_PATH   = os.path.join(DATA_PATH, 'Images')
MASQUES_PATH  = os.path.join(DATA_PATH, 'Masques')
SRC_PATH      = os.path.join(PROJECT_PATH, 'src')
OUTPUTS_PATH  = os.path.join(PROJECT_PATH, 'outputs')

sys.path.append(SRC_PATH)
os.makedirs(OUTPUTS_PATH, exist_ok=True)

print('📂 Contenu ISIC_Project :')
for f in sorted(os.listdir(PROJECT_PATH)):
    print(f'   {f}')

## 📦 1.3 — Installation des dépendances

In [ ]:
!pip install -q opencv-python-headless albumentations scikit-learn matplotlib seaborn tqdm pillow
print('✅ Dépendances installées !')

## 📥 1.4 — Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
import random
import seaborn as sns
from tqdm import tqdm
import torch

print(f'🖥️  PyTorch : {torch.__version__}')
print(f'🖥️  GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Non disponible"}')

## 🔍 1.5 — Chargement des fichiers (filtre images uniquement)

In [ ]:
# ✅ CORRECTION : filtre les fichiers non-images (ATTRIBUTION.txt, etc.)
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

images_list  = sorted([p for p in Path(IMAGES_PATH).glob('*.*')
                       if p.suffix.lower() in IMAGE_EXTENSIONS])
masques_list = sorted([p for p in Path(MASQUES_PATH).glob('*.*')
                       if p.suffix.lower() in IMAGE_EXTENSIONS])

print(f'🖼️  Images  : {len(images_list)}')
print(f'🎭 Masques : {len(masques_list)}')

# Vérification correspondance
assert len(images_list) == len(masques_list), \
    f'❌ Nombre images ({len(images_list)}) != masques ({len(masques_list)})'
print('✅ Correspondance images/masques OK !')

# Afficher les fichiers ignorés
ignored_imgs  = [p.name for p in Path(IMAGES_PATH).glob('*.*')
                 if p.suffix.lower() not in IMAGE_EXTENSIONS]
ignored_masks = [p.name for p in Path(MASQUES_PATH).glob('*.*')
                 if p.suffix.lower() not in IMAGE_EXTENSIONS]
if ignored_imgs:  print(f'⚠️  Ignorés dans Images/  : {ignored_imgs}')
if ignored_masks: print(f'⚠️  Ignorés dans Masques/ : {ignored_masks}')

## 📐 1.6 — Statistiques des données

In [ ]:
sizes   = [Image.open(p).size for p in tqdm(images_list[:50], desc='Lecture tailles')]
widths  = [s[0] for s in sizes]
heights = [s[1] for s in sizes]

print(f'\n📐 Largeur  : min={min(widths)}, max={max(widths)}, moy={int(np.mean(widths))}')
print(f'📐 Hauteur  : min={min(heights)}, max={max(heights)}, moy={int(np.mean(heights))}')
print(f'📄 Exemples : {[p.name for p in images_list[:3]]}')

## 🖼️ 1.7 — Visualisation Images / Masques / Overlay

In [ ]:
def show_samples(images_list, masques_list, n=4):
    indices = random.sample(range(len(images_list)), min(n, len(images_list)))
    fig, axes = plt.subplots(3, n, figsize=(4*n, 12))
    for i, idx in enumerate(indices):
        img     = np.array(Image.open(images_list[idx]).convert('RGB'))
        msk     = np.array(Image.open(masques_list[idx]).convert('L'))
        overlay = img.copy()
        overlay[msk > 127] = [255, 100, 100]
        axes[0, i].imshow(img);              axes[0, i].set_title(f'Image {idx}');  axes[0, i].axis('off')
        axes[1, i].imshow(msk, cmap='gray'); axes[1, i].set_title(f'Masque {idx}'); axes[1, i].axis('off')
        axes[2, i].imshow(overlay);          axes[2, i].set_title('Overlay');       axes[2, i].axis('off')
    plt.suptitle('Exemples ISIC — Image / Masque / Overlay', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_PATH, 'samples_visualization.png'), dpi=150)
    plt.show()
    print('💾 Sauvegardé : outputs/samples_visualization.png')

show_samples(images_list, masques_list)

## 📊 1.8 — Distribution de la surface lésionnelle

In [ ]:
ratios = []
for mp in tqdm(masques_list[:100], desc='Analyse masques'):
    msk = np.array(Image.open(mp).convert('L'))
    ratios.append((msk > 127).sum() / msk.size * 100)

plt.figure(figsize=(10, 4))
plt.hist(ratios, bins=30, color='steelblue', edgecolor='white')
plt.xlabel('% pixels lésion')
plt.ylabel("Nombre d'images")
plt.title('Distribution de la surface de lésion (%)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_PATH, 'mask_distribution.png'), dpi=150)
plt.show()
print(f'\n📊 Surface moyenne : {np.mean(ratios):.1f}%')
print(f'📊 Surface min     : {min(ratios):.1f}%')
print(f'📊 Surface max     : {max(ratios):.1f}%')

## ✅ Résumé
- Drive connecté et chemins définis
- Fichiers non-images ignorés automatiquement (ex: ATTRIBUTION.txt)
- Données explorées et visualisées
- Visualisations sauvegardées dans `outputs/`

➡️ **Prochain notebook : `02_Preprocessing_et_Dataset.ipynb`**